# 03 Evaluation: Base vs Fine-Tuned Model

This notebook compares the baseline FLAN-T5 model against the LoRA fine-tuned adapter on the **same fixed test subset**.

In [ ]:
# imports
import sys
import random
import importlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import src.model as model_module
import src.evaluate as eval_module
importlib.reload(model_module)
importlib.reload(eval_module)

from src.model import DomainSummarizer, GenerationParams
from src.evaluate import compute_rouge_batch, format_comparison_row, to_markdown_table

_ = (
    datetime,
    timezone,
    pd,
    tqdm,
    load_dataset,
    DomainSummarizer,
    GenerationParams,
    compute_rouge_batch,
    format_comparison_row,
    to_markdown_table,
)

RESULTS_FILE = ROOT / "results" / "comparison_table.md"
print(f"Root: {ROOT}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Results file: {RESULTS_FILE}")

Root: d:\GenAI_DeepLearning\domain-summarizer
Device: cuda
Results file: d:\GenAI_DeepLearning\domain-summarizer\results\comparison_table.md


In [ ]:
# configurations
BASE_MODEL_NAME = "google/flan-t5-base"
ADAPTER_PATH = ROOT / "lora-adapter"

NUM_SAMPLES = 100

generation_cfg = GenerationParams(
    temperature=0.7,
    top_p =0.95,
    max_new_tokens=128,
)

if not ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"Adapter directory not found at {ADAPTER_PATH}. Run notebook 02 training first."
    )

print(f"Base model: {BASE_MODEL_NAME}")
print(f"Sample count: {NUM_SAMPLES}")
print(f"Generation config: {generation_cfg}")

Base model: google/flan-t5-base
Sample count: 100
Generation config: GenerationParams(temperature=0.7, top_k=None, top_p=0.95, max_new_tokens=128)


In [ ]:
# load same test subset
print("Loading test split...")
test_ds = load_dataset("cnn_dailymail", "3.0.0", split="test")
test_subset = test_ds.shuffle(seed=SEED).select(range(NUM_SAMPLES))

articles = [x["article"] for x in test_subset]
references = [x["highlights"] for x in test_subset]
print(f"Loaded {len(articles)} evaluation samples.")

Loading test split...
Loaded 100 evaluation samples.


In [ ]:
# load models
base_model = DomainSummarizer(model_name=BASE_MODEL_NAME, seed=SEED)
ft_model = DomainSummarizer(model_name=BASE_MODEL_NAME, adapter_path=str(ADAPTER_PATH), seed=SEED)

base_model.load()
ft_model.load()
print("Both models loaded.")

base_predictions = []
ft_predictions = []
base_errors = 0
ft_errors = 0

for article in tqdm(articles, desc="Evaluating base vs fine-tuned"):
    try:
        base_pred = base_model.summarize(article=article, generation=generation_cfg)
    except Exception as exc:
        base_pred = ""
        base_errors += 1
        print(f"Base generation error: {exc}")

    try:
        ft_pred = ft_model.summarize(article=article, generation=generation_cfg)
    except Exception as exc:
        ft_pred = ""
        ft_errors += 1
        print(f"Fine-tuned generation error: {exc}")

    base_predictions.append(base_pred)
    ft_predictions.append(ft_pred)

base_metrics = compute_rouge_batch(references, base_predictions)
ft_metrics = compute_rouge_batch(references, ft_predictions)

comparison_rows = [
    format_comparison_row(
        model_label="FLAN-T5 Base (Post-Train Eval)",
        metrics=base_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"temp={generation_cfg.temperature}, errors={base_errors}",
    ),
    format_comparison_row(
        model_label="FLAN-T5 + LoRA (Fine-tuned)",
        metrics=ft_metrics,
        sample_count=NUM_SAMPLES,
        notes=f"adapter=lora-adapter, temp={generation_cfg.temperature}, errors={ft_errors}",
    ),
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Both models loaded.


Evaluating base vs fine-tuned:   0%|          | 0/100 [00:00<?, ?it/s]

,model,rouge1,rouge2,rougeL,samples,notes
0,FLAN-T5 Base (Post-Train Eval),0.3389,0.1203,0.2321,100,"temp=0.7, errors=0"
1,FLAN-T5 + LoRA (Fine-tuned),0.3645,0.1413,0.2479,100,"adapter=lora-adapter, temp=0.7, errors=0"


In [ ]:
# show improvements
improvement = {
    "rouge1_delta": round(ft_metrics["rouge1"] - base_metrics["rouge1"], 4),
    "rouge2_delta": round(ft_metrics["rouge2"] - base_metrics["rouge2"], 4),
    "rougeL_delta": round(ft_metrics["rougeL"] - base_metrics["rougeL"], 4),
}

improvement_df = pd.DataFrame([improvement])
improvement_df

,rouge1_delta,rouge2_delta,rougeL_delta
0,0.0256,0.021,0.0158


In [ ]:
# save results to markdown file
eval_table_md = to_markdown_table(comparison_rows)
run_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

existing = RESULTS_FILE.read_text(encoding="utf-8") if RESULTS_FILE.exists() else "# Baseline vs Fine-Tuned ROUGE Comparison\n"
marker = "## Fine-Tuned Evaluation (After LoRA)"
if marker in existing:
    existing = existing.split(marker)[0].rstrip() + "\n\n"

append_block = "\n".join([
    marker,
    "",
    f"Generated at: {run_time}",
    "",
    eval_table_md,
    "",
    "## Improvement Summary (Fine-tuned - Base)",
    f"- ROUGE-1 delta: {improvement['rouge1_delta']:+.4f}",
    f"- ROUGE-2 delta: {improvement['rouge2_delta']:+.4f}",
    f"- ROUGE-L delta: {improvement['rougeL_delta']:+.4f}",
])

RESULTS_FILE.write_text(existing + append_block + "\n", encoding="utf-8")
print(f"Updated {RESULTS_FILE}")

Updated d:\GenAI_DeepLearning\domain-summarizer\results\comparison_table.md


In [ ]:
# side-by-side preview of first few samples
preview_rows = []
for i in range(min(3, NUM_SAMPLES)):
    preview_rows.append(
        {
            "sample_idx": i,
            "reference": references[i][:250],
            "base": base_predictions[i][:250],
            "fine_tuned": ft_predictions[i][:250],
        }
    )

pd.DataFrame(preview_rows)

,sample_idx,reference,base,fine_tuned
0,0,CNN's Dr. Sanjay Gupta says we should legalize...,Marijuana is a medicine that can be used to tr...,"Marijuana is on the rise in the United States,..."
1,1,Child has amassed thousands of Twitter followe...,"Boy, from Memphis, Tennessee, has amassed more...","Little boy, from Memphis, Tennessee, has been ..."
2,2,The presidential hopeful held a town hall meet...,New Jersey Governor Chris Christie is being ca...,New Jersey governor Chris Christie gets into h...


In [ ]:
# cleanup
base_model.unload()
ft_model.unload()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup complete.")

Cleanup complete.
